# Foundations NLP Block Course Assignment

Train and evaluate character-level GPT model on small Shakespeare dataset using nanoGPT

Student Names: Alexis Liston
Course: Foundations of NLP
WiSe 25/26

## Notebook Outline
1. Environment setup and verification
2. Baseline trianing
3. Experiment with hyperparameters
4. Log parsing, plots
5. Samples and qualitative evaluation
6. Analysis

### 1. Environment Setup and Verification

In [ ]:
# verify environment setup

import sys
import torch
import numpy as np

print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (PyTorch): {torch.version.cuda}")

Python version: 3.11.15
PyTorch version: 2.11.0+cu130
NumPy version: 2.4.3
CUDA available: True
CUDA device: NVIDIA RTX A1000 6GB Laptop GPU
CUDA version (PyTorch): 13.0


In [ ]:
# verify correct working directory and folders

import os
print(f"Current working directory: {os.getcwd()}")
print(f"train.py exists: {os.path.exists('train.py')}")
print(f"data/shakespeare_char/train.bin exists: {os.path.exists('data/shakespeare_char/train.bin')}")
print(f"logs/ exists: {os.path.exists('logs')}")
print(f"config/ exists: {os.path.exists('config')}")

Current working directory: c:\Users\lexil\OneDrive\Uni Osnabruck\WiSe 25-26\NLP Block Courses\NLP Foundations Assignment\FoundationsNLP
train.py exists: True
data/shakespeare_char/train.bin exists: True
logs/ exists: True
config/ exists: True


### 2. Baseline Training

baseline training as specified in assignment. I'm using an NVIDIA RTX A100 GPU, so will set device = 'cuda'

In [3]:
%%writefile config/train_shakespeare_char_baseline.py
# baseline config file specified in assignment

out_dir = 'out-shakespeare-baseline'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False
wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'baseline'
dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100
weight_decay = 1e-1
device = 'cuda' # using 'cuda' with GPU
compile = False # compiler can have issues on Windows; leaving False for stability for now

Writing config/train_shakespeare_char_baseline.py


In [6]:
# run the baseline training
# pipe output to the notebook display and a log file (baseline.log). "!" doesn't mix well with PowerShell, so I'll try letting python handle it with subprocess. 

import subprocess
import time

log_path = "logs/baseline.log"
cmd = ["python", "train.py", "config/train_shakespeare_char_baseline.py"]

start_time = time.time()

with open(log_path, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")          # live display in the notebook
        log_file.write(line)         # simultaneously write to log file
        log_file.flush()             # flush so the file updates in real time
    process.wait()

elapsed = time.time() - start_time
minutes, seconds = divmod(elapsed, 60)

summary = (
    f"\n--- Training finished with exit code {process.returncode} ---\n"
    f"Wall-clock time: {int(minutes)} min {seconds:.1f} sec ({elapsed:.1f} s total)\n"
)
print(summary)

# Append the timing to the log file so it's preserved alongside the training output
with open(log_path, "a", encoding="utf-8") as log_file:
    log_file.write(summary)

Overriding config with config/train_shakespeare_char_baseline.py:
# baseline config file specified in assignment

out_dir = 'out-shakespeare-baseline'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False
wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'baseline'
dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100
weight_decay = 1e-1
device = 'cuda' # using 'cuda' with GPU
compile = False # compiler can have issues on Windows; leaving False for stability for now

tokens per iteration will be: 16,384
found vocab_size = 65 (inside data\shakespeare_char\meta.pkl)
Initializing a new model from scratch
number of parameters: 10.65M
c:\Users\lexil\OneDrive\Uni Osnabruck\WiSe 25-26\NLP Block Courses\NLP Foundations Assignment\FoundationsNLP\tra

In [7]:
# Sample generation from the trained model

import subprocess

sample_output_path = "logs/baseline_samples.txt"
cmd = [
    "python", "sample.py",
    "--out_dir=out-shakespeare-baseline",
    "--device=cuda",
    "--num_samples=3",
    "--max_new_tokens=500",
]

with open(sample_output_path, "w", encoding="utf-8") as sample_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        sample_file.write(line)
        sample_file.flush()
    process.wait()

print(f"\n--- Sampling finished with exit code {process.returncode} ---")
print(f"Samples saved to: {sample_output_path}")

Overriding: out_dir = out-shakespeare-baseline
Overriding: device = cuda
Overriding: num_samples = 3
Overriding: max_new_tokens = 500
number of parameters: 10.65M
Loading meta from data\shakespeare_char\meta.pkl...


ANGELO:
And cowards it be strawn'd; and you will grave you
gates your happiness, that by the shorten with me.

ISABELLA:
In gold time of it.

DUKE VINCENTIO:
Why, well, how that Hercules the former properioved me
else no more.

ANGELO:
Here's a minister.

ISABELLA:
I will answer the prisoner of the
most Capulet, prize on the dominion. I have
strange it for the king thrust for the great sheep-by allowers, and
she was a marble soldier.

ANGELO:
Had you a help?

ISABELLA:
A neither to change me to
---------------

Men pardon me, you shall have hanged to myself.

LORD FITZWATER:
The honour is so not unborn, whom comes off
Which mighty more should perjury of evils.

WARWICK:
Hence stays our father; but what makes the grounds
Of blood, against his sovereign news our household.

